# Milestone 3 — Alzheimer's Detection Data Pipeline

**Detecting Alzheimer's Disease from Clinical Discharge Notes using Pre-Trained Transformer Models**

Group 18 · Vinay Manikandan Nagarajan

This notebook runs the full data pipeline on **MIMIC-IV** discharge notes and is
tuned for a **Colab T4 GPU**. Set `Runtime → Change runtime type → T4 GPU`, then
run every cell top to bottom.

Pipeline: Drive → cohort (ICD) → filtered note read → preprocess → split →
frozen Bio_ClinicalBERT embeddings → TF-IDF baseline → train + evaluate.


## 1. Check the GPU
Confirm a T4 is attached before doing any heavy work.


In [ ]:
!nvidia-smi

## 2. Get the project code
If you cloned the repo into Colab, point `REPO_DIR` at it. Otherwise set your
GitHub URL below and this cell clones it. The cell then `cd`s to the repo root
(the folder containing `config/config.yaml`).


In [ ]:
import os, sys

# Option A: repo already present (uploaded or cloned). Set the path:
REPO_DIR = '/content/alzheimers-clinical-nlp'

# Option B: clone from GitHub (uncomment and set your URL):
# if not os.path.isdir(REPO_DIR):
#     !git clone $GITHUB_URL $REPO_DIR

# Fallback: if this notebook lives inside the repo, walk up to find config/
if not os.path.isfile(os.path.join(REPO_DIR, 'config', 'config.yaml')):
    here = os.getcwd()
    while here != '/' and not os.path.isfile(os.path.join(here,'config','config.yaml')):
        here = os.path.dirname(here)
    REPO_DIR = here

assert os.path.isfile(os.path.join(REPO_DIR,'config','config.yaml')), \
    'Could not locate the repo. Set REPO_DIR to the folder containing config/config.yaml.'
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Repo root:', REPO_DIR)

## 3. Install dependencies
Colab already ships a CUDA-matched `torch` — **do not reinstall it**. We only
add `transformers` (and pin the light deps just in case).


In [ ]:
!pip -q install 'transformers>=4.38' 'pyarrow>=14.0' 'pyyaml>=6.0' tqdm joblib
print('deps ready')

## 4. Mount Google Drive
The pipeline reads the MIMIC parquet files from
`My Drive/ADRD_IDR/mimic/`. Adjust `paths.drive_root` in `config/config.yaml`
if your folder differs.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Quick check that the folder is visible:
from src.config import load_config
cfg = load_config('config/config.yaml')
print('Looking in:', cfg.paths.drive_root)
!ls -lh "{cfg.paths.drive_root}" | head

## 5. (Optional) tune for your runtime
Defaults are conservative for a free T4. If the runtime is stable you can
speed things up by raising the batch size; if you hit OOM, lower it.


In [ ]:
# Uncomment to override without editing the YAML:
# cfg.embeddings.batch_size = 32   # faster, more VRAM
# cfg.cohort.control_ratio  = 2    # smaller dataset, quicker run
print('batch_size =', cfg.embeddings.batch_size,
      '| control_ratio =', cfg.cohort.control_ratio,
      '| fp16 =', cfg.embeddings.fp16)

## 6. Run the full pipeline
This executes all seven stages. The 1.74 GB notes file is read with a filtered
scan, and embeddings are extracted in fp16 with small batches so the T4 is not
overwhelmed. Embeddings are cached to `artifacts/` — if the kernel restarts,
re-running skips the expensive recomputation.


In [ ]:
from src.run_pipeline import run

# do_mount=False because we already mounted Drive in step 4.
results = run(config_path='config/config.yaml', do_mount=False)

## 7. Inspect results
Metrics for both models (Bio_ClinicalBERT vs TF-IDF), on validation and test.


In [ ]:
import json, pandas as pd
with open('results/metrics.json') as f:
    m = json.load(f)
pd.DataFrame(m).T.round(4)

### Plots


In [ ]:
from IPython.display import Image, display
import glob, os
for p in sorted(glob.glob('results/*.png')):
    print(os.path.basename(p))
    display(Image(p))

## 8. Where things landed
- `artifacts/` — cohort, labelled dataset, cached embeddings
- `models/` — trained logistic-regression models + TF-IDF vectorizer
- `results/` — `metrics.json` + confusion-matrix and ROC plots

These feed the next milestone (feature attribution for RQ2 and the generative
explanation module for RQ3).
